<a href="https://colab.research.google.com/github/Desire-in-tech/Customer-Segmentation/blob/main/02_clustering_two_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2: K-Means Clustering — Two Features

**Dataset:** Survey of Consumer Finances (SCF) 2019  
**Method:** K-Means clustering using two features (Home Value & Mortgage Debt)

In this notebook we:
- Build a `wrangle` function for the TURNFEAR subset
- Construct a 2-feature matrix X
- Fit K-Means and extract cluster labels and centroids
- Evaluate with inertia and silhouette score
- Find the optimal K using elbow and silhouette curves
- Visualize the final clusters

In [1]:
#Clone the GitHub repository
REPO_URL = 'https://github.com/Desire-in-tech/Customer-Segmentation.git'

!git clone {REPO_URL}

import os

os.chdir('/content/Customer-Segmentation')

#Install required packages
!pip install pandas numpy plotly scikit-learn scipy --quiet

print(f'Working directory: {os.getcwd()}')
print(f'Data file exists: {os.path.exists("data/SCF_2019.csv.gz")}')

fatal: destination path 'Customer-Segmentation' already exists and is not an empty directory.
Working directory: /content/Customer-Segmentation
Data file exists: True


## 1. Imports

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '${:,.2f}'.format)

print('Libraries loaded!')

Libraries loaded!


## 2. Wrangle Function

In [3]:
def wrangle(filepath):
    """
    Load and wrangle SCF 2019 data.

    Parameters
    ----------
    filepath : str
        Path to the compressed CSV file (SCF_2019.csv.gz)

    Returns
    -------
    pd.DataFrame
        Subset of households that were turned down for credit
        or feared being denied credit (TURNFEAR == 1).
        Uses implicate 1 only (every 5th row).
    """
    # Load data
    df = pd.read_csv(filepath, compression='gzip', index_col=0)

    # Keep implicate 1 (every 5th row)
    df = df.iloc[::5].reset_index(drop=True)

    # Subset: households turned down or fearing credit denial
    mask = df['TURNFEAR'] == 1
    df = df[mask].copy().reset_index(drop=True)

    return df


# Load the data
df = wrangle('data/SCF_2019.csv.gz')

print(f'Shape: {df.shape}')
print(f'Households with TURNFEAR == 1: {len(df):,}')
df.head()

Shape: (926, 357)
Households with TURNFEAR == 1: 926


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,...,NWCAT,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT
0,2,21,"$3,790.48",1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
1,23,231,"$5,424.99",1,69,5,11,3,2,0,...,1,1,1,2,1,3,2,3,1,1
2,31,311,"$4,730.10",1,31,1,9,3,1,2,...,1,3,2,3,2,1,6,6,3,3
3,37,371,"$4,705.03",1,45,3,2,1,1,0,...,1,2,1,2,1,3,4,4,2,2
4,40,401,"$5,620.47",1,68,5,12,4,2,0,...,2,2,3,2,1,5,4,3,2,2


## 3. Feature Selection: Two Features

We'll use **Home Value** (`HOUSES`) and **Mortgage Debt** (`MRTHEL`) as our two clustering features.

In [4]:
# Feature matrix with two features
features = ['HOUSES', 'MRTHEL']

X = df[features].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'\nFeature descriptions:')
print(f'  HOUSES  — Home/primary residence value')
print(f'  MRTHEL  — Mortgage debt on primary residence')
print()
X.describe()

Feature matrix shape: (926, 2)

Feature descriptions:
  HOUSES  — Home/primary residence value
  MRTHEL  — Mortgage debt on primary residence



,HOUSES,MRTHEL
count,$926.00,$926.00
mean,"$217,868.33","$95,816.20"
std,"$1,514,519.74","$745,551.77"
min,$0.00,$0.00
25%,$0.00,$0.00
50%,$0.00,$0.00
75%,"$139,104.64","$23,184.11"
max,"$37,175,713.91","$20,865,695.36"


## 4. Build Initial K-Means Model (K=3)

In [5]:
# Build and fit K-Means model with K=3
model = KMeans(n_clusters=3, random_state=42, n_init=10)
model.fit(X)

print('K-Means model fitted successfully!')
print(f'Number of clusters: {model.n_clusters}')
print(f'Number of iterations: {model.n_iter_}')

K-Means model fitted successfully!
Number of clusters: 3
Number of iterations: 2


## 5. Extract Cluster Labels

In [6]:
# Extract cluster labels
labels = model.labels_

print('Cluster label distribution:')
label_counts = pd.Series(labels).value_counts().sort_index()
for cluster, count in label_counts.items():
    print(f'  Cluster {cluster}: {count:,} households ({count/len(labels)*100:.1f}%)')

# Add labels to X
X_labeled = X.copy()
X_labeled['Cluster'] = labels.astype(str)

Cluster label distribution:
  Cluster 0: 923 households (99.7%)
  Cluster 1: 1 households (0.1%)
  Cluster 2: 2 households (0.2%)


## 6. Scatter Plot of Clusters

In [7]:
# Filter extremes for visualization
X_plot = X_labeled[
    (X_labeled['HOUSES'] < X_labeled['HOUSES'].quantile(0.98)) &
    (X_labeled['MRTHEL'] < X_labeled['MRTHEL'].quantile(0.98))
].copy()

fig = px.scatter(
    X_plot,
    x='HOUSES',
    y='MRTHEL',
    color='Cluster',
    title='K-Means Clustering: Home Value vs. Mortgage Debt (K=3)',
    labels={'HOUSES': 'Home Value ($)', 'MRTHEL': 'Mortgage Debt ($)', 'Cluster': 'Cluster'},
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig.show()

## 7. Extract and Plot Centroids

In [8]:
import plotly.graph_objects as go

# Extract centroids
centroids = model.cluster_centers_
centroids_df = pd.DataFrame(centroids, columns=features)
centroids_df.index.name = 'Cluster'
centroids_df.index = [f'Cluster {i}' for i in range(len(centroids_df))]

print('Cluster Centroids:')
print(centroids_df.to_string())

# Plot clusters + centroids
# Using X_labeled directly to include all points without filtering extremes
fig = px.scatter(
    X_labeled,
    x='HOUSES',
    y='MRTHEL',
    color='Cluster',
    title='K-Means Clusters with Centroids (K=3)',
    labels={'HOUSES': 'Home Value ($)', 'MRTHEL': 'Mortgage Debt ($)'},
    opacity=0.5,
    color_discrete_sequence=px.colors.qualitative.Set1
)

# Get the color map from px.scatter for consistent coloring
color_map = {cluster_id: color for cluster_id, color in zip(sorted(X_labeled['Cluster'].unique()), px.colors.qualitative.Set1)}

# Add centroids as larger markers with matching cluster colors
for i, centroid in enumerate(centroids):
    cluster_label = str(i) # Cluster labels are strings in X_labeled
    fig.add_trace(
        go.Scatter(
            x=[centroid[0]],
            y=[centroid[1]],
            mode='markers',
            marker=dict(size=18, symbol='x', color=color_map.get(cluster_label, 'black'), line_width=3),
            name=f'Centroid {cluster_label}',
            showlegend=False # Avoid duplicate legends for centroids if cluster legends are already shown
        )
    )
fig.show()

Cluster Centroids:
                  HOUSES         MRTHEL
Cluster 0    $141,488.68     $63,172.54
Cluster 1 $37,175,713.91 $20,865,695.36
Cluster 2 $16,988,153.64  $4,775,925.83


## 8. Model Evaluation: Inertia and Silhouette Score

In [9]:
# Calculate inertia
inertia = model.inertia_

# Calculate silhouette score
sil_score = silhouette_score(X, labels)

print(f'K-Means Model Evaluation (K=3):')
print(f'  Inertia:         {inertia:,.2f}')
print(f'  Silhouette Score: {sil_score:.4f}  (range: -1 to 1, higher is better)')

K-Means Model Evaluation (K=3):
  Inertia:         225,965,946,613,110.59
  Silhouette Score: 0.9822  (range: -1 to 1, higher is better)


## 9. Finding Optimal K: Elbow and Silhouette Curves

In [10]:
# Test K from 2 to 12
k_range = range(2, 13)
inertia_errors = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertia_errors.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X, km.labels_))
    print(f'K={k:2d} | Inertia: {km.inertia_:>18,.0f} | Silhouette: {silhouette_score(X, km.labels_):.4f}')

print('\nDone!')

K= 2 | Inertia: 670,244,795,402,594 | Silhouette: 0.9867
K= 3 | Inertia: 225,965,946,613,111 | Silhouette: 0.9822
K= 4 | Inertia: 105,437,737,342,801 | Silhouette: 0.9203
K= 5 | Inertia: 50,603,202,192,100 | Silhouette: 0.8647
K= 6 | Inertia: 31,136,044,168,054 | Silhouette: 0.7905
K= 7 | Inertia: 21,967,183,289,047 | Silhouette: 0.7939
K= 8 | Inertia: 16,635,703,528,877 | Silhouette: 0.7935
K= 9 | Inertia: 13,907,465,490,552 | Silhouette: 0.7854
K=10 | Inertia: 11,678,855,922,912 | Silhouette: 0.7838
K=11 | Inertia:  9,723,353,660,801 | Silhouette: 0.7820
K=12 | Inertia:  8,172,258,439,095 | Silhouette: 0.7802

Done!


## 10. Elbow Plot: Inertia vs. Number of Clusters

In [11]:
fig = px.line(
    x=list(k_range),
    y=inertia_errors,
    markers=True,
    title='K-Means Model: Inertia vs Number of Clusters',
    labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'}
)
fig.update_traces(line_color='#636EFA', marker_color='#EF553B', marker_size=8)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=2, dtick=1))
fig.show()

## 11. Silhouette Score vs. Number of Clusters

In [12]:
fig = px.line(
    x=list(k_range),
    y=silhouette_scores,
    markers=True,
    title='K-Means Model: Silhouette Score vs Number of Clusters',
    labels={'x': 'Number of Clusters (K)', 'y': 'Silhouette Score'}
)
fig.update_traces(line_color='#00CC96', marker_color='#EF553B', marker_size=8)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=2, dtick=1))
fig.show()

best_k = list(k_range)[np.argmax(silhouette_scores)]
print(f'Best K by silhouette score: K={best_k} (score={max(silhouette_scores):.4f})')

Best K by silhouette score: K=2 (score=0.9867)


## 12. Final Model

Based on the elbow and silhouette plots, choose the best K.

In [13]:
# Build final model with the best K (adjust based on your plots)
BEST_K = 4

final_model = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
final_model.fit(X)

final_labels = final_model.labels_
final_inertia = final_model.inertia_
final_sil = silhouette_score(X, final_labels)

print(f'Final Model (K={BEST_K})')
print(f'  Inertia:          {final_inertia:,.2f}')
print(f'  Silhouette Score: {final_sil:.4f}')

label_counts = pd.Series(final_labels).value_counts().sort_index()
print(f'\nCluster sizes:')
for cluster, count in label_counts.items():
    print(f'  Cluster {cluster}: {count:,} ({count/len(final_labels)*100:.1f}%)')

Final Model (K=4)
  Inertia:          105,437,737,342,801.17
  Silhouette Score: 0.9203

Cluster sizes:
  Cluster 0: 14 (1.5%)
  Cluster 1: 1 (0.1%)
  Cluster 2: 2 (0.2%)
  Cluster 3: 909 (98.2%)


## 13. Final Cluster Scatter Plot

In [15]:
import plotly.graph_objects as go

X_final = X.copy()
X_final['Cluster'] = final_labels.astype(str)

# Get the final centroids from the final model
final_centroids = final_model.cluster_centers_

# Using X_final directly to include all points without filtering extremes
fig = px.scatter(
    X_final,
    x='HOUSES',
    y='MRTHEL',
    color='Cluster',
    title=f'Final K-Means Clusters: Home Value vs. Mortgage Debt (K={BEST_K})',
    labels={'HOUSES': 'Home Value ($)', 'MRTHEL': 'Mortgage Debt ($)'},
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Plotly
)

# Get the color map from px.scatter for consistent coloring
color_map_final = {cluster_id: color for cluster_id, color in zip(sorted(X_final['Cluster'].unique()), px.colors.qualitative.Plotly)}

# Add centroids as larger markers with matching cluster colors
for i, centroid in enumerate(final_centroids):
    cluster_label = str(i) # Cluster labels are strings in X_final
    fig.add_trace(
        go.Scatter(
            x=[centroid[0]],
            y=[centroid[1]],
            mode='markers',
            marker=dict(size=18, symbol='x', color=color_map_final.get(cluster_label, 'black'), line_width=3),
            name=f'Centroid {cluster_label}',
            showlegend=False # Avoid duplicate legends for centroids if cluster legends are already shown
        )
    )
fig.show()

## 14. Side-by-Side Bar Chart: Mean Features by Cluster

In [16]:
# Compute mean of the two features per cluster
X_final_means = X.copy()
X_final_means['Cluster'] = final_labels
cluster_means = X_final_means.groupby('Cluster')[features].mean().reset_index()
cluster_means['Cluster'] = cluster_means['Cluster'].astype(str)

# Melt for grouped bar chart
cluster_means_melted = cluster_means.melt(
    id_vars='Cluster',
    value_vars=features,
    var_name='Feature',
    value_name='Mean Value'
)

fig = px.bar(
    cluster_means_melted,
    x='Cluster',
    y='Mean Value',
    color='Feature',
    barmode='group',
    title=f'Mean Feature Values by Cluster (K={BEST_K})',
    labels={'Cluster': 'Cluster', 'Mean Value': 'Mean Value ($)', 'Feature': 'Feature'},
    color_discrete_sequence=px.colors.qualitative.Plotly
)
fig.show()

print('Mean values by cluster:')
print(cluster_means.to_string(index=False))

Mean values by cluster:
Cluster         HOUSES         MRTHEL
      0  $2,938,585.43    $872,384.79
      1 $37,175,713.91 $20,865,695.36
      2 $16,988,153.64  $4,775,925.83
      3     $98,409.08     $50,709.42


## Concluding Analysis: K-Means Customer Segmentation

This notebook aimed to segment households (specifically those who were turned down for credit or feared being denied credit) based on two key financial features: **Home Value (`HOUSES`)** and **Mortgage Debt (`MRTHEL`)** using K-Means clustering.

### Optimal K Selection

While the silhouette score suggested K=2 as the optimal number of clusters (with a score of 0.9867), this often indicates that the data has a very clear primary separation, potentially overlooking meaningful, albeit smaller, distinctions. The elbow method, though less definitive, showed a more gradual curve. For this analysis, we proceeded with `BEST_K = 4`, indicating a decision to explore more granular segments within the dataset.

### Final Clustering Results (K=4)

With `K=4`, the K-Means algorithm identified four distinct clusters. Their characteristics, based on mean Home Value and Mortgage Debt, are as follows:

*   **Cluster 3 (909 households, 98.2% of the data):**
    *   **Mean Home Value:** $98,409.08
    *   **Mean Mortgage Debt:** $50,709.42
    *   **Interpretation:** This is by far the largest cluster, representing the vast majority of households. These individuals tend to have relatively low home values and correspondingly low mortgage debt. Given the `TURNFEAR` context, this group might be struggling with credit access despite their modest asset and debt levels, or their perceived fear of denial is not directly tied to high property values or debt.

*   **Cluster 0 (14 households, 1.5% of the data):**
    *   **Mean Home Value:** $2,938,585.43
    *   **Mean Mortgage Debt:** $872,384.79
    *   **Interpretation:** This cluster consists of a small group of households with significant home values and substantial mortgage debt, but not at the extreme highest end. They represent a segment of affluent homeowners who, despite their wealth in property, are still part of the `TURNFEAR` group. Their credit concerns might stem from the sheer scale of their debt or other undisclosed financial complexities.

*   **Cluster 2 (2 households, 0.2% of the data):**
    *   **Mean Home Value:** $16,988,153.64
    *   **Mean Mortgage Debt:** $4,775,925.83
    *   **Interpretation:** An even smaller cluster of very high-value homeowners with correspondingly very high mortgage debt. This group represents an extremely wealthy segment facing credit access issues, possibly due to unique financial structures or high-risk investments associated with their substantial assets and liabilities.

*   **Cluster 1 (1 household, 0.1% of the data):**
    *   **Mean Home Value:** $37,175,713.91
    *   **Mean Mortgage Debt:** $20,865,695.36
    *   **Interpretation:** This cluster is an extreme outlier, consisting of a single household with exceptionally high home value and mortgage debt. This individual's credit challenges are likely unique and may not be representative of broader trends, possibly indicative of highly specialized financial situations.

### Key Findings

1.  **Dominance of Low-Value Households:** The overwhelming majority (98.2%) of households in the `TURNFEAR` subset fall into a segment characterized by relatively low home values and mortgage debt. This suggests that credit access issues or fear of denial are not solely a problem for those with high levels of debt or large assets. Other factors beyond these two features are likely at play for this large group.
2.  **Presence of Affluent Segments:** Despite being a small fraction of the `TURNFEAR` population, there are distinct clusters of very affluent households with high home values and significant mortgage debt who also experience credit access concerns. This highlights that `TURNFEAR` is not exclusive to lower-income or lower-asset individuals.
3.  **Outliers and Data Skewness:** The presence of extremely high-value clusters (especially Cluster 1) indicates a significant skew in the data, where a few individuals hold disproportionately high assets and debts. This can impact clustering results and emphasizes the importance of understanding the data distribution.

### Limitations

The current analysis is limited to two features (`HOUSES` and `MRTHEL`). While these provide initial insights, a more comprehensive segmentation would likely require incorporating additional demographic, income, and debt-related features to provide a richer understanding of why these diverse groups experience credit concerns. The choice of `K=4` revealed these distinct groups, even if K=2 offered a higher silhouette score, demonstrating the trade-off between statistical purity and granular interpretability.